In [1]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np

from Background_functions import read_audio_file, split_audio_segments, start_end_times
from STFT import short_time_calc, plot_spectrogram
from DFT import dtw_calc

from pathlib import Path

DATA_22 = Path(r"C:\Users\HP\Desktop\Skripsie data\DataSubmission\2023_04_22")

def load_file_calls(wav_path, text_path):
    f_s, x = read_audio_file(wav_path)
    _, start_t, end_t = start_end_times(text_path)
    segments = split_audio_segments(x, f_s, start_t, end_t)
    return f_s, x, start_t, end_t, segments

In [2]:
def select_calls_interactive(segments, f_s, start_t, audio_template,
                              expected_count, n_templates, n_calibration,
                              call_type="single_tone", file_label="",
                              exclude_idx=None):
    """
    segments        : whale_call_segments for this file
    f_s              : sample rate for this file
    start_t          : start times for this file (for reporting only)
    audio_template   : example audio snippet of the call type you're selecting
    expected_count   : how many of this call type you expect in this file (from your table)
    n_templates      : how many approved calls to allocate to templates
    n_calibration    : how many approved calls to allocate to calibration
    exclude_idx      : indices to skip entirely (e.g. the template call itself)
    Returns a dict that gets filled in as you click through (see note below).
    """
    exclude_idx = set(exclude_idx or [])

    f_temp, t_temp, Zxx_temp, fs_new = short_time_calc(audio_template, f_s)

    num_calls = len(segments)
    stft_cost = np.zeros(num_calls)
    for i in range(num_calls):
        _, _, Zxx_seg, _ = short_time_calc(segments[i], f_s)
        stft_cost[i] = dtw_calc(Zxx_temp, Zxx_seg)

    sorted_idx = np.argsort(stft_cost)
    sorted_idx = np.array([i for i in sorted_idx if i not in exclude_idx])
    candidate_idx = sorted_idx[:expected_count]

    n_total = n_templates + n_calibration
    approved_idx = []
    state = {"pos": 0}
    results = {"template_idx": [], "calibration_idx": [],
               "call_type": call_type, "file_label": file_label,
               "stft_cost": stft_cost, "done": False}

    out = widgets.Output()
    status = widgets.Label()
    btn_yes = widgets.Button(description="Keep (y)", button_style="success")
    btn_no = widgets.Button(description="Reject (n)", button_style="danger")
    btn_stop = widgets.Button(description="Stop / Finish", button_style="warning")
    display(widgets.HTML(f"<b>{file_label} — {call_type}</b>"), status,
            widgets.HBox([btn_yes, btn_no, btn_stop]), out)

    def show_next():
        out.clear_output(wait=True)
        if len(approved_idx) >= n_total or state["pos"] >= len(candidate_idx):
            finalize()
            return
        idx = int(candidate_idx[state["pos"]])
        with out:
            f_sig, t_sig, Zxx_sig, fsn = short_time_calc(segments[idx], f_s)
            plot_spectrogram(f_sig, t_sig, Zxx_sig, fsn)
            print(f"rank {state['pos']}/{len(candidate_idx)-1} | "
                  f"cost {stft_cost[idx]:.4f} | index {idx} | start_t {start_t[idx]:.2f}")
        status.value = f"Approved: {len(approved_idx)}/{n_total}"

    def finalize():
        slots = np.unique(np.linspace(0, len(approved_idx) - 1,
                           min(n_templates, len(approved_idx))).round().astype(int)) \
                 if approved_idx else np.array([], dtype=int)
        results["template_idx"] = [approved_idx[i] for i in slots]
        results["calibration_idx"] = [approved_idx[i] for i in range(len(approved_idx))
                                       if i not in slots]
        results["done"] = True
        out.clear_output(wait=True)
        with out:
            print(f"Done. Templates: {len(results['template_idx'])}, "
                  f"Calibration: {len(results['calibration_idx'])}")
        btn_yes.disabled = btn_no.disabled = btn_stop.disabled = True

    def on_yes(b):
        approved_idx.append(int(candidate_idx[state["pos"]]))
        state["pos"] += 1
        show_next()

    def on_no(b):
        state["pos"] += 1
        show_next()

    def on_stop(b):
        finalize()

    btn_yes.on_click(on_yes)
    btn_no.on_click(on_no)
    btn_stop.on_click(on_stop)
    show_next()
    return results

In [10]:
def package_results(res, start_t, end_t):
    tmpl = res["template_idx"]
    calib = res["calibration_idx"]
    return {
        "call_type": res["call_type"], "file": res["file_label"],
        "template_idx": tmpl, "template_start_t": start_t[tmpl], "template_end_t": end_t[tmpl],
        "calibration_idx": calib, "calibration_start_t": start_t[calib], "calibration_end_t": end_t[calib],
    }

Recording: 20230422_171301

In [9]:
f_s, x, start_t, end_t, segments = load_file_calls(
    DATA_22 / "20230422_171301.WAV",
    DATA_22 / "20230422_171301.Detections.selections.txt"
)

Single-tones (20230422_171301)

In [ ]:
audio_template_1 = segments[14]

st_results = select_calls_interactive(
    segments, f_s, start_t, audio_template_1,
    expected_count=259, n_templates=25, n_calibration=155,
    call_type="single_tone", file_label="20230422",
    exclude_idx=[14]
)

HTML(value='<b>20230422 — single_tone</b>')

Label(value='')

Output()

In [11]:
all_results = []   # master list across every file/call type you run
all_results.append(package_results(st_results, start_t, end_t))

In [12]:
print("Template indices:", st_results["template_idx"])
print("Calibration indices:", st_results["calibration_idx"])

Template indices: [20, 149, 29, 605, 34, 148, 69, 600, 132, 209, 338, 595, 348, 472, 333, 510, 513, 144, 398, 49, 475, 307, 368, 451, 216]
Calibration indices: [618, 171, 32, 159, 46, 109, 592, 226, 26, 227, 40, 126, 150, 215, 401, 103, 30, 44, 24, 61, 623, 458, 383, 193, 80, 17, 85, 201, 499, 174, 35, 58, 152, 454, 114, 379, 436, 81, 37, 175, 413, 173, 370, 211, 86, 31, 606, 514, 495, 449, 182, 371, 374, 526, 620, 284, 156, 218, 282, 168, 319, 244, 42, 367, 361, 341, 71, 164, 198, 161, 375, 186, 79, 240, 163, 94, 523, 299, 258, 462, 145, 245, 247, 194, 214, 415, 476, 141, 407, 373, 67, 446, 57, 323, 428, 508, 613, 322, 420, 503, 478, 310, 372, 208, 504, 519, 347, 426, 72, 480, 60, 277, 200, 362, 596, 291, 122, 237, 516, 395, 421, 471, 521, 627, 365, 68, 488, 507, 179, 312, 517, 155, 74, 135, 378, 305, 316, 108, 0, 329, 98, 357, 496, 327, 301, 309, 443, 105, 180, 512, 1, 381, 143, 112, 390]


Multi-tones (20230422_171301)